# Evaluation Code Review Extraction

Notebook created by Johanna Mauermann.

The code is slightly adapted from [this notebook](https://github.com/johannamauermann/LLM_Biases_Article_Extraction/blob/main/Evaluation_code_article_extraction.ipynb), with the difference of extracting reviews instead of articles.

---

This notebook contains code to evaluate the performance of Large Language Models (LLMs) in a text extraction task by comparing their output to a manually curated ground truth. For each review identified by the LLM, the code determines a match status and calculates a success score. It categorizes outcomes into True Positives, False Positives, False Negatives, and True Negatives, while identifying specific issues such as Over-Extraction or Under-Extraction (when the LLM extracts too much or too little text), Content Mismatch (indicated by a high Levenshtein distance), and Formatting Errors (when text boundaries differ from the ground truth).

**Context**: master thesis titled "Literaturbesprechungen in historischen Zeitungen: Korpusbildung und Analyse diachroner Entwicklungen mit LLMs und NLP-Methoden"

## Main function for matching reviews

This function tries to find matches between ground truth and LLM extracted reviews.

In [ ]:
# main function for alignment
import pandas as pd
!pip install fuzzywuzzy
!pip install Python-Levenshtein
from fuzzywuzzy import fuzz
import Levenshtein
import re

N = 50
threshold = 65

def match_reviews(row, llm_column, gt_column='reviews', N=50, threshold=65):


    results = []

    gt_text = row[gt_column]
    llm_text = row[llm_column]


    if pd.isna(gt_text) or pd.isna(llm_text):
        append_result(results,
            aligned=False,
            match_status="Incomplete Data",
            start_matched_text=None,
            end_matched_text=None,
            distance_to_llm_start=None,
            distance_to_llm_end=None,
            lev_distance=None,
            normalized_lev_distance=None,
            normalized_omitted_characters=None,
            normalized_added_characters=None,
            human_verification_needed=False,
            success_score=None
        )
        return results

    ground_truth_list = transform_to_dict(gt_text)
    llm_list = transform_to_dict(llm_text)


    pattern = r"\s*.{0,20}(No reviews found|No review found|\[\])"

    gt_matches = [gt_dict for gt_dict in ground_truth_list if re.match(pattern, gt_dict.get("review", ""))]
    llm_matches = [llm_dict for llm_dict in llm_list if re.match(pattern, llm_dict.get("review", ""))]


    if gt_matches and llm_matches:
        append_result(results,
            aligned=False,
            match_status="True Negative",
            start_matched_text=None,
            end_matched_text=None,
            distance_to_llm_start=0,
            distance_to_llm_end=0,
            lev_distance=0,
            normalized_lev_distance=0,
            normalized_omitted_characters=0,
            normalized_added_characters=0,
            human_verification_needed=False,
            success_score=1
        )
        return results

    if not gt_matches and llm_matches:
        for gt_dict in ground_truth_list:
            append_result(results,
                aligned=False,
                match_status="False Negative",
                start_matched_text=None,
                end_matched_text=None,
                distance_to_llm_start=None,
                distance_to_llm_end=None,
                lev_distance=None,
                normalized_lev_distance=None,
                normalized_omitted_characters=None,
                normalized_added_characters=None,
                human_verification_needed=False,
                success_score=0
            )
        return results

    if gt_matches and not llm_matches:
        for llm_dict in llm_list:
            append_result(results,
                aligned=False,
                match_status="False Positive",
                start_matched_text=None,
                end_matched_text=None,
                distance_to_llm_start=None,
                distance_to_llm_end=None,
                lev_distance=None,
                normalized_lev_distance=None,
                normalized_omitted_characters=None,
                normalized_added_characters=None,
                human_verification_needed=llm_dict.get("human_verification_needed", "Unclear"),
                success_score=0
            )
        return results

    all_matched_reviews_gt = []
    all_matched_reviews_llm = []

    if len(llm_list) <= len(ground_truth_list):


        for llm_dict in llm_list:

            matched_reviews_gt = []

            llm_text = llm_dict.get("review", "")

            for gt_dict in ground_truth_list:
                gt_text = gt_dict.get("review", "")

                matching = review_matches(gt_text, llm_text)

                if matching:
                    matched_reviews_gt.append(gt_text)
                    all_matched_reviews_gt.append(gt_text)


            formatting_error = False

            if len(matched_reviews_gt) == 0:
                append_result(results,
                    aligned=False,
                    match_status="False Positive",
                    start_matched_text=None,
                    end_matched_text=None,
                    distance_to_llm_start=None,
                    distance_to_llm_end=None,
                    lev_distance=None,
                    normalized_lev_distance=None,
                    normalized_omitted_characters=None,
                    normalized_added_characters=None,
                    human_verification_needed=llm_dict.get("human_verification_needed", "Unclear"),
                    success_score=0
                )
                return results

            if len(matched_reviews_gt) > 1:
                formatting_error = True

            gt_text = " ".join(matched_reviews_gt)

            gt_start_snippet = get_character_snippet(gt_text, 0, N)
            gt_end_snippet = get_character_snippet(gt_text, -N, N)

            (
                start_position,
                start_score,
                start_matched_text,
                distance_to_llm_start,
                end_position,
                end_score,
                end_matched_text,
                distance_to_llm_end
            ) = calculate_start_end_distances(
                gt_text, llm_text,
                gt_start_snippet,
                gt_end_snippet,
                N,
                threshold
            )


            if distance_to_llm_start is not None and distance_to_llm_end is not None:
                lev_distance, normalized_lev_distance = calculate_levenshtein(
                    gt_text, llm_text,
                    distance_to_llm_start,
                    distance_to_llm_end
                )
            else:
                lev_distance = None
                normalized_lev_distance = None

            normalized_omitted_characters, normalized_added_characters = normalize_extraction(
                distance_to_llm_start,
                distance_to_llm_end,
                gt_text,
                llm_text
            )

            success_score = calculate_success_score(
                normalized_lev_distance,
                normalized_omitted_characters,
                normalized_added_characters,
                formatting_error
            )

            match_status = determine_match_status(
                normalized_lev_distance,
                normalized_omitted_characters,
                normalized_added_characters,
                formatting_error
            )

            append_result(results,
                aligned=True,
                match_status=match_status,
                start_matched_text=start_matched_text,
                end_matched_text=end_matched_text,
                distance_to_llm_start=distance_to_llm_start,
                distance_to_llm_end=distance_to_llm_end,
                lev_distance=lev_distance,
                normalized_lev_distance=normalized_lev_distance,
                normalized_omitted_characters=normalized_omitted_characters,
                normalized_added_characters=normalized_added_characters,
                human_verification_needed=llm_dict.get("human_verification_needed", "Unclear"),
                success_score=success_score
            )

    if len(llm_list) > len(ground_truth_list):

        for gt_dict in ground_truth_list:

            matched_reviews_llm = []

            gt_text = gt_dict.get("review", "")

            for llm_dict in llm_list:
                llm_text = llm_dict.get("review", "")

                matching = review_matches(gt_text, llm_text)

                if matching:
                    matched_reviews_llm.append(llm_text)
                    all_matched_reviewsllm.append(llm_text)


            formatting_error = False

            if len(matched_reviews_llm) == 0:
                append_result(results,
                    aligned=False,
                    match_status="False Negative",
                    start_matched_text=None,
                    end_matched_text=None,
                    distance_to_llm_start=None,
                    distance_to_llm_end=None,
                    lev_distance=None,
                    normalized_lev_distance=None,
                    normalized_omitted_characters=None,
                    normalized_added_characters=None,
                    human_verification_needed=None,
                    success_score=0
                )
                return results

            if len(matched_reviews_llm) > 1:
                formatting_error = True

            llm_text = " ".join(matched_reviews_llm)

            gt_start_snippet = get_character_snippet(gt_text, 0, N)
            gt_end_snippet = get_character_snippet(gt_text, -N, N)

            (
                start_position,
                start_score,
                start_matched_text,
                distance_to_llm_start,
                end_position,
                end_score,
                end_matched_text,
                distance_to_llm_end
            ) = calculate_start_end_distances(
                gt_text, llm_text,
                gt_start_snippet,
                gt_end_snippet,
                N,
                threshold
            )

            if distance_to_llm_start is not None and distance_to_llm_end is not None:
                lev_distance, normalized_lev_distance = calculate_levenshtein(
                    gt_text, llm_text,
                    distance_to_llm_start,
                    distance_to_llm_end
                )
            else:
                lev_distance = None
                normalized_lev_distance = None

            normalized_omitted_characters, normalized_added_characters = normalize_extraction(
                distance_to_llm_start,
                distance_to_llm_end,
                gt_text,
                llm_text
            )

            success_score = calculate_success_score(
                normalized_lev_distance,
                normalized_omitted_characters,
                normalized_added_characters,
                formatting_error
            )

            match_status = determine_match_status(
                normalized_lev_distance,
                normalized_omitted_characters,
                normalized_added_characters,
                formatting_error
            )

            append_result(results,
                aligned=True,
                match_status=match_status,
                start_matched_text=start_matched_text,
                end_matched_text=end_matched_text,
                distance_to_llm_start=distance_to_llm_start,
                distance_to_llm_end=distance_to_llm_end,
                lev_distance=lev_distance,
                normalized_lev_distance=normalized_lev_distance,
                normalized_omitted_characters=normalized_omitted_characters,
                normalized_added_characters=normalized_added_characters,
                human_verification_needed=None,
                success_score=success_score
            )

    return results

## Helper functions to match snippets

In [ ]:
# helper function for fuzzy matching
def best_fuzzy_match_char(target_snippet, search_text, N, threshold):
    """
    Finds the best fuzzy match for a target snippet within the search text.
    Handles cases where search text is shorter than target snippet.
    """

    #target_snippet = target_snippet.lower()
    #search_text = search_text.lower()

    best_score = 0
    best_position = None

    # If search text is shorter than N characters, compare with entire search text
    if len(search_text) < N:
        score = fuzz.partial_ratio(target_snippet, search_text)
        if score >= threshold:
            return 0, score
        return None, 0

    # For longer texts, slide the window
    for i in range(len(search_text) - N + 1):
        window_text = search_text[i:i + N]
        score = fuzz.partial_ratio(target_snippet, window_text)

        if score > best_score and score >= threshold:
            best_score = score
            best_position = i

    # If we found no match above threshold, try one last time with the full text
    if best_position is None:
        full_score = fuzz.partial_ratio(target_snippet, search_text)
        if full_score >= threshold:
            return 0, full_score

    return best_position, best_score

# Helper function to make matching more robust
def get_character_snippet(text, position, N):
    """
    Extracts a snippet of N characters from a text starting at a given position.
    Handles cases where text is shorter than N characters.
    """
    if len(text) < N:
        return text  # Return entire text if shorter than N

    if position >= 0:
        return text[position:position + N]
    else:
        # For negative positions, ensure we get exactly N characters if possible
        start = max(len(text) + position, 0)  # Avoid negative start index
        return text[start:start + N]


## Helper function: transform_to_dict

*   This function converts text, either in a single string format or an XML format with review tags, into a standardized dictionary format. When using the tag format, the dictionary also includes the review's human_verification_needed information (if it exists). Additionally, this function is planned to support various output formats beyond just tags in the future.
*   The function "normalize_text" is called to remove unneccessary line breaks and white spaces



In [ ]:
import re

def transform_to_dict(text):
    dict_list = []

    # Patterns for extracting reviewss and human verification
    review_pattern = r'<review>\s*([^<>\s].*?)\s*</review>'
    verification_pattern = r'<human_verification_needed>(.*?)</human_verification_needed>'
    irrelevant_pattern = r"No reviews found|No review found|\[\]|<review></review>|<review>...</review>)"
    # Extract all reviews and verification tags
    review_matches = re.findall(review_pattern, text, re.DOTALL | re.IGNORECASE)
    verification_matches = re.findall(verification_pattern, text, re.DOTALL)
    irrelevant_matches = re.findall(irrelevant_pattern, text, re.DOTALL)

    if irrelevant_matches and not review_matches:
      dict_list.append({
        'review': 'No review found.',
        'all_reviews_on_page': normalize_text(text.strip()),
        'human_verification_needed': None
      })
      return dict_list

    # edge case handling for cases with mismatched open and closing tags
    open_tags = len(re.findall(r"<review>", text))
    close_tags = len(re.findall(r"</review>", text))

    if open_tags != close_tags and "<review></review>" in text:
          dict_list.append({
            'review': 'No review found.',
            'all_reviews_on_page': normalize_text(text.strip()),
            'human_verification_needed': None
          })
          return dict_list

    # Normalize and combine all reviews into a single string
    combined_review = normalize_text(" ".join(review_matches).strip()) if review_matches else None

    # Ensure the verification list matches the review list in length
    if len(review_matches) > len(verification_matches):
        verification_matches.extend([None] * (len(review_matches) - len(verification_matches)))
    elif len(review_matches) < len(verification_matches):
        verification_matches = verification_matches[:len(review_matches)]

    # Create a list of dictionaries for each review
    for i, review_text in enumerate(review_matches):
        human_verification = None
        if verification_matches[i]:
            verification_value = verification_matches[i].strip().replace('**', '').lower()
            human_verification = True if verification_value == 'true' else False if verification_value == 'false' else None

        # Normalize the review text
        normalized_review = normalize_text(review_text)

        dict_list.append({
            'review': normalized_review,
            'all_reviews_on_page': combined_review,
            'human_verification_needed': human_verification
        })

    # Handle case where no <review> tags are found
    if not review_matches:
        dict_list.append({
            'review': normalize_text(text.strip()),
            'all_reviews_on_page': normalize_text(text.strip()),
            'human_verification_needed': None
        })

    return dict_list

def normalize_text(text):
    text = re.sub(r'<title>.*?</title>', '', text)
    text = re.sub(r'<headline>.*?</headline>', '', text)

    # Remove specific tags: <title>, <content>, <paragraph>
    text = re.sub(r'</?(content|paragraph|contenu|section|inhalt)>', '', text)


    # Normalize whitespace: replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'<review>`-Tags', '', text)
    text = re.sub(r'<review>` tags', '', text)

    # Remove all line breaks
    text = text.replace('\n', ' ')

    # Remove multiple spaces
    text = ' '.join(text.split())

    # Remove spaces before punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    return text


## Helper function: review_matches

This helper function is called iteratively in the main function. For each review, it determines if there is a match by trying to align start and end snippets. If a match is found, it returns "True", otherwise "False".

In [ ]:
def review_matches(gt_text, llm_text):
    # Ground truth snippets
    gt_start_snippet = get_character_snippet(gt_text, 0, N)
    gt_end_snippet = get_character_snippet(gt_text, -N, N)

    # Forward search in LLM text
    start_position, start_score = best_fuzzy_match_char(gt_start_snippet, llm_text, N, threshold)
    end_position, end_score = best_fuzzy_match_char(gt_end_snippet, llm_text, N, threshold)

    rev_start_position, rev_start_score = None, None
    rev_end_position, rev_end_score = None, None

    # Handle start position and distance
    if start_position is None:
        # Reverse search from LLM to GT, if forward search fails
        rev_start_position, rev_start_score = best_fuzzy_match_char(
            get_character_snippet(llm_text, 0, N),
            gt_text,
            N,
            threshold
        )

    if end_position is None:
        # Reverse search from end, if forward search fails
        llm_end_snippet = get_character_snippet(llm_text, -N, N)
        rev_end_position, rev_end_score = best_fuzzy_match_char(llm_end_snippet, gt_text, N, threshold)

    # Evaluate results
    start_match = start_position is not None or rev_start_position is not None
    end_match = end_position is not None or rev_end_position is not None

    return start_match and end_match


## Helper function: calculate_start_end_distances

For matched reviews, this function determines whether and by how many characters the start and end of the Ground Truth review differ from the LLM review. A negative value indicates omitted characters, while a positive value signifies added characters.


In [ ]:
def calculate_start_end_distances(gt_text, llm_text, gt_start_snippet, gt_end_snippet, N, threshold):
    start_matched_text, end_matched_text = None, None
    distance_to_llm_start, distance_to_llm_end = None, None

    # Forward search
    start_position, start_score = best_fuzzy_match_char(gt_start_snippet, llm_text, N, threshold)
    end_position, end_score = best_fuzzy_match_char(gt_end_snippet, llm_text, N, threshold)

    # Reverse search if not found
    if start_position is None:
        rev_start_position, rev_start_score = best_fuzzy_match_char(get_character_snippet(llm_text, 0, N), gt_text, N, threshold)
        if rev_start_position is not None:
            start_matched_text = gt_text[rev_start_position:rev_start_position + N]
            start_score = rev_start_score
            distance_to_llm_start = -rev_start_position
    else:
        start_matched_text = llm_text[start_position:start_position + N]
        distance_to_llm_start = start_position

    if end_position is None:
        llm_end_snippet = get_character_snippet(llm_text, -N, N)
        rev_end_position, rev_end_score = best_fuzzy_match_char(llm_end_snippet, gt_text, N, threshold)
        if rev_end_position is not None:
            end_matched_text = gt_text[rev_end_position:rev_end_position + N]
            end_score = rev_end_score
            distance_to_llm_end = -(len(gt_text) - (rev_end_position + N))
    else:
        end_matched_text = llm_text[end_position:end_position + N]
        llm_distance_from_end = len(llm_text) - (end_position + N)
        gt_distance_from_end = 0  # GT ends here, as it's the gold standard
        distance_to_llm_end = llm_distance_from_end - gt_distance_from_end

    return (
        start_position,
        start_score,
        start_matched_text,
        distance_to_llm_start,
        end_position,
        end_score,
        end_matched_text,
        distance_to_llm_end
    )


## Helper Function: append_result

Appends a dictionary, the values depending on the calculated metrics for the respective review.

In [ ]:
def append_result(results, aligned, match_status, start_matched_text, end_matched_text, distance_to_llm_start, distance_to_llm_end, lev_distance, normalized_lev_distance, normalized_omitted_characters, normalized_added_characters, human_verification_needed, success_score):
    results.append({
        'aligned': aligned,
        'match_status': match_status,
        'start_matched_text': start_matched_text,
        'end_matched_text': end_matched_text,
        'start_distance': distance_to_llm_start,
        'end_distance': distance_to_llm_end,
        'levenshtein_distance': lev_distance,
        'normalized_levenshtein': normalized_lev_distance,
        'normalized_omitted_characters': normalized_omitted_characters,
        'normalized_added_characters': normalized_added_characters,
        'human_verification_needed': human_verification_needed,
        'success_score': success_score
    })


## Helper function: calculate_success_score

Starting from the perfect score 1, this function calculates a score for all aligned reviews by subtracting relative to the *normalized_lev_distance*, *normalized_omitted_characters*, and *normalized_added_characters* values (weighing omitted characters x1,5, as even a very small percentage of missing characters can be a significant issue).

For the Levenshtein distance, we apply a tolerance threshold of 0.1, allowing minor adjustments such as corrections of OCR mistakes.

In [ ]:
def calculate_success_score(normalized_lev_distance, normalized_omitted_characters, normalized_added_characters, formatting_error):
  success_score = 1
  ocr_tolerance = 0.1 # or another value that makes sense in your context
  # Check if normalized_lev_distance is None
  if normalized_lev_distance is not None:
    adjusted_lev = max(0, normalized_lev_distance - ocr_tolerance)
  else:
    adjusted_lev = 0
  if formatting_error:
    success_score -= 0.2 # if there's a formatting error, we substract 20%
  success_score -= (adjusted_lev + (1.5 *normalized_omitted_characters) + normalized_added_characters) # percentual substractions: omitted characters weighted more heavily because they are the most severe issue

  return max(0, success_score)

## Helper function: determine_match_status

This function differentiates between True Positive and other _aligned=True_ cases that have some issues: Over-Extraction, Under-Extraction, Content-Mismatch, Formatting Error.

In [ ]:
def determine_match_status(normalized_lev_distance, normalized_omitted_characters, normalized_added_characters, formatting_error):
    # Determine individual statuses
    over_extraction = normalized_added_characters > 0.04
    under_extraction = normalized_omitted_characters > 0.04
    content_mismatch = normalized_lev_distance > 0.2

    # Collect active statuses
    active_statuses = []
    if over_extraction:
        active_statuses.append("Over-Extraction")
    if under_extraction:
        active_statuses.append("Under-Extraction")
    if content_mismatch:
        active_statuses.append("Content Mismatch")
    if formatting_error:
        active_statuses.append("Formatting Error")

    # Determine final match_status
    if not active_statuses:
        match_status = "True Positive"  # If no errors are detected #Fehler wahrscheinlich hier - active_statuses falsch detektiert
    elif len(active_statuses) == 1:
        match_status = active_statuses[0]  # If only one status is active, use string
    else:
        match_status = active_statuses  # If multiple statuses, use list

    return match_status


In [ ]:
def normalize_extraction(distance_to_llm_start, distance_to_llm_end, gt_text, llm_text):
    distances = [distance_to_llm_start, distance_to_llm_end]
    total_omitted_characters = sum(abs(d) for d in distances if d is not None and d < 0)
    total_added_characters = sum(d for d in distances if d is not None and d > 0)
    normalized_omitted_characters = total_omitted_characters / len(gt_text) if len(gt_text) > 0 else 0
    normalized_added_characters = total_added_characters / len(llm_text) if len(llm_text) > 0 else 0
    return normalized_omitted_characters, normalized_added_characters


## Helper function to calculate Levenshtein Distance

In [ ]:
def calculate_levenshtein(gt_text, llm_text, distance_to_llm_start, distance_to_llm_end):
    gt_start_idx = 0 if distance_to_llm_start >= 0 else abs(distance_to_llm_start)
    gt_end_idx = len(gt_text) - (abs(distance_to_llm_end) if distance_to_llm_end < 0 else 0)

    llm_start_idx = distance_to_llm_start if distance_to_llm_start > 0 else 0
    llm_end_idx = len(llm_text) - (abs(distance_to_llm_end) if distance_to_llm_end > 0 else 0)

    gt_between_text = gt_text[gt_start_idx:gt_end_idx]
    llm_between_text = llm_text[llm_start_idx:llm_end_idx]
    lev_distance = Levenshtein.distance(gt_between_text, llm_between_text)
    max_length = max(len(gt_between_text), len(llm_between_text))
    normalized_lev_distance = lev_distance / max_length if max_length > 0 else 0
    return lev_distance, normalized_lev_distance


## Function: Summarize_success


Returns a dictionary, containing:
*   average success score, both page wise and review wise
*   counters for different match statuses (when more than one match status applies, the review gets added to the "multiple issues" counter)
*  counters for *human_verification_needed* and *incomplete_data* (those reviews are excluded from the calculation as they require human review)
*  lists containing the relevant metrics for reviews with issues, enabling to later review severity
* percentages for relevant as well as non-relevant reviews
* percentage of perfect results

In [ ]:
results = summarize_success(df, evaluation_result_column="insert_column_name")

for key, value in results.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for sub_key, sub_value in value.items():
            print(f"  {sub_key}: {sub_value}")
    else:
        print(f"{key}: {value}")

In [ ]:
import ast
import pandas as pd
import re

def get_status_from_value(status):
    """Convert status value to standard format"""
    if isinstance(status, list):
        return status
    return status

def count_reviews_in_ground_truth(text):
    """Count number of <review> tags in ground truth text"""
    return len(re.findall(r'<review>.*?</review>', text, re.DOTALL))

def summarize_success(df, evaluation_result_column, ground_truth_column="ground_truth"):
    """
    Summarize success metrics for review extraction and matching.

    Args:
        df: DataFrame containing evaluation results
        evaluation_result_column: Name of column containing evaluation dictionaries
        ground_truth_column: Name of column containing ground truth text

    Returns:
        Dictionary containing various success metrics
    """
    scores_page = []
    scores_review = []
    total_valid_pages = 0

    counters = {
        "true_positive": 0,
        "true_negative": 0,
        "false_negative": 0,
        "false_positive": 0,
        "formatting_error": 0,
        "under_extraction": 0,
        "over_extraction": 0,
        "content_mismatch": 0,
        "multiple_issues": 0,
        "needs_human_verification": 0,
        "incomplete_data": 0,
    }

    under_extraction_scores = []
    over_extraction_scores = []
    content_mismatch_scores = []
    multiple_issue_review_scores = []
    needs_human_verification_scores = []

    if isinstance(df[evaluation_result_column].iloc[0], str):
        df[evaluation_result_column] = df[evaluation_result_column].apply(ast.literal_eval)

    for _, row in df.iterrows():
        eval_dicts = row[evaluation_result_column]
        page_score = 0
        valid_entries = 0

        if eval_dicts is None or not isinstance(eval_dicts, list) or pd.isna(eval_dicts).any():
            continue

        # Check if any review in this row needs human verification
        has_human_verification = any(
            eval_dict.get("human_verification_needed", False)
            for eval_dict in eval_dicts
        )

        for eval_dict in eval_dicts:
            raw_match_status = eval_dict.get("match_status", "Unknown")
            match_status = get_status_from_value(raw_match_status)
            aligned = eval_dict.get("aligned", False)

            human_verification_needed = eval_dict.get("human_verification_needed", False)
            review_score = eval_dict.get("success_score", None)

            if human_verification_needed:
                counters["needs_human_verification"] += 1
                if review_score is not None:
                    needs_human_verification_scores.append(review_score)
                continue  # Skip reviews that need human verification

            # Count found reviews - not just aligned ones
            if isinstance(match_status, list):
                base_statuses = match_status
            else:
                base_statuses = [match_status]

            for status in base_statuses:
                if status == "True Positive":
                    counters["true_positive"] += 1
                elif status == "True Negative":
                    counters["true_negative"] += 1
                elif status == "False Negative":
                    counters["false_negative"] += 1
                elif status == "False Positive":
                    counters["false_positive"] += 1
                elif status == "Formatting Error":
                    counters["formatting_error"] += 1
                elif status == "Under-Extraction":
                    counters["under_extraction"] += 1
                    if not isinstance(match_status, list):
                        under_extraction_scores.append(eval_dict.get("normalized_omitted_characters", ""))
                elif status == "Over-Extraction":
                    counters["over_extraction"] += 1
                    if not isinstance(match_status, list):
                        over_extraction_scores.append(eval_dict.get("normalized_added_characters", ""))
                elif status == "Content Mismatch":
                    counters["content_mismatch"] += 1
                    if not isinstance(match_status, list):
                        content_mismatch_scores.append(eval_dict.get("normalized_levenshtein", ""))

            if match_status == "Incomplete Data":
                counters["incomplete_data"] += 1
                continue

            # Score calculations
            if review_score is not None:
                page_score += review_score
                valid_entries += 1
                scores_review.append(review_score)

                if isinstance(match_status, list):
                    counters["multiple_issues"] += 1
                    multiple_issue_review_scores.append(review_score)

        if valid_entries > 0:
            scores_page.append(page_score / valid_entries)
            total_valid_pages += 1

    # Calculate final metrics
    try:
        average_score_pages = sum(scores_page) / total_valid_pages if total_valid_pages > 0 else 0
    except ZeroDivisionError:
        average_score_pages = 0

    try:
        average_score_reviews = sum(scores_review) / len(scores_review) if scores_review else 0
    except ZeroDivisionError:
        average_score_reviews = 0

    # Calculate total_relevant_base (excluding reviews needing human verification)
    total_relevant_base = (
        counters["true_positive"] + counters["false_negative"] + counters["formatting_error"] +
        counters["under_extraction"] + counters["over_extraction"] + counters["content_mismatch"] +
        counters["multiple_issues"]
    )

    # Calculate relevant percentages
    relevant_percentages = {
        "true_positive": (counters["true_positive"] / total_relevant_base) if total_relevant_base > 0 else 0,
        "false_negative": (counters["false_negative"] / total_relevant_base) if total_relevant_base > 0 else 0,
        "formatting_error": (counters["formatting_error"] / total_relevant_base) if total_relevant_base > 0 else 0,
        "under_extraction": (counters["under_extraction"] / total_relevant_base) if total_relevant_base > 0 else 0,
        "over_extraction": (counters["over_extraction"] / total_relevant_base) if total_relevant_base > 0 else 0,
        "content_mismatch": (counters["content_mismatch"] / total_relevant_base) if total_relevant_base > 0 else 0,
        "multiple_issues": (counters["multiple_issues"] / total_relevant_base) if total_relevant_base > 0 else 0,
    }

    # Calculate total_relevant_percentage (without false negatives)
    total_relevant_percentage = (
        relevant_percentages["true_positive"] +
        relevant_percentages["formatting_error"] +
        relevant_percentages["under_extraction"] +
        relevant_percentages["over_extraction"] +
        relevant_percentages["content_mismatch"] +
        relevant_percentages["multiple_issues"]
    )

    # Add total_relevant_percentage to relevant_percentages
    relevant_percentages["total_relevant_percentage"] = total_relevant_percentage
    perfect_proportion = relevant_percentages["true_positive"] / total_relevant_percentage

    # Calculate total_non_relevant
    total_non_relevant = counters["true_negative"] + counters["false_positive"]

    # Calculate non-relevant percentages
    non_relevant_percentages = {
        "true_negative": (counters["true_negative"] / total_non_relevant) if total_non_relevant > 0 else 0,
        "false_positive": (counters["false_positive"] / total_non_relevant) if total_non_relevant > 0 else 0,
    }

    return {
        "average_score_pages": average_score_pages,
        "average_score_reviews": average_score_reviews,
        "counters": counters,
        "all_page_scores": scores_page,
        "all_review_scores": scores_review,
        "under_extraction_scores (normalized omitted characters)": under_extraction_scores,
        "over_extraction_scores (normalized added characters)": over_extraction_scores,
        "content_mismatch_scores (normalized lev. distance)": content_mismatch_scores,
        "multiple_issue_review_scores": multiple_issue_review_scores,
        "needs_human_verification_review_scores": needs_human_verification_scores,
        "relevant_percentages": relevant_percentages,
        "non_relevant_percentages": non_relevant_percentages,
        "perfect_proportion": perfect_proportion
    }